<a href="https://colab.research.google.com/github/segomezz/Practica-1-IA/blob/Segomezz-Ontolog%C3%ADas/Pr%C3%A1ctica1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2.3. Ontología y Razonamiento Semántico (RDFLib y OWL-RL)

In [14]:
!pip install rdflib
!pip install owlrl
!pip install scikit-fuzzy
!pip install experta

  Preparing metadata (setup.py) ... done
  Created wheel for frozendict: filename=frozendict-1.2-py3-none-any.whl size=3149 sha256=e35f22ec7158d6936d5a9493137a53f898b4abd6852779e1fd9c69ebd5b1d4ac
  Stored in directory: /root/.cache/pip/wheels/49/ac/f8/cb8120244e710bdb479c86198b03c7b08c3c2d3d2bf448fd6e
Successfully built frozendict
  Attempting uninstall: frozendict
    Found existing installation: frozendict 2.4.6
    Uninstalling frozendict-2.4.6:
      Successfully uninstalled frozendict-2.4.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
yfinance 0.2.57 requires frozendict>=2.3.4, but you have frozendict 1.2 which is incompatible.


In [15]:
from rdflib import Graph, Namespace, RDF, RDFS, Literal, XSD, URIRef
from rdflib.namespace import FOAF, DC
from owlrl import DeductiveClosure, RDFS_Semantics
import collections.abc

if not hasattr(collections, 'Mapping'):
    collections.Mapping = collections.abc.Mapping
from experta import *


* Implementación de la Ontología del dominio utilizando RDF y RDFS.
  * El lenguaje de serialización de la ontología final desarrollada será en
Turtle.

  * Implementar al menos 10 clases (rdfs:Class) y 5 relaciones jerárquicas
(rdfs:subClassOf).

A continuación se realiza la definición de un diccionario llamada clases en el cual se establecen clase : subclases. Luego ese se recorre y se le da a cada clase y super clase su URI

In [ ]:
# Namespaces
INV = Namespace("http://example.org/inversiones#")

# Crear el grafo
g = Graph()
g.bind("inv", INV)
g.bind("foaf", FOAF)
g.bind("dc", DC)

# Clases (10 clases + jerarquías)
clases = {
    "Inversion": None,
    "Activo": "Inversion",
    "Riesgo": "Inversion",
    "PerfilInversionista": "foaf:Person",
    "Recomendacion": None,
    "Accion": "Activo",
    "Bonos": "Activo",
    "Criptoactivo": "Activo",
    "InversionConservadora": "Inversion",
    "InversionAgresiva": "Inversion",
}

for clase, superclase in clases.items():
    clase_uri = INV[clase]
    g.add((clase_uri, RDF.type, RDFS.Class))
    if superclase:
        if ":" in superclase:
            prefix, name = superclase.split(":")
            super_uri = FOAF[name] if prefix == "foaf" else DC[name]
        else:
            super_uri = INV[superclase]
        g.add((clase_uri, RDFS.subClassOf, super_uri))



* Crear al menos 10 propiedades RDF originales con dominios
(rdfs:domain) y rangos (rdfs:range) correctamente definidos.
    * Incluir al menos un caso de jerarquía (rdfs:subPropertyOf)
    * Incluir como rangos valores tanto de tipo clase (URIs) y otros Tipo
Literales con sus respectivos tipados usando el XML-Schema.

  * Incluir el uso de clases o propiedades provenientes de vocabularios definidos y aceptados en la literatura (FOAF, Dublin Core, etc.).
  *Instanciar al menos 4 individuos por clase

  Se crea un Diccionario con las 10 propiedades y una subpropiedad. Acada propiedad se le asigna su Uri y de igual manera para el rango y el dominio

  Finalmente se realizan las 4 instancias

In [ ]:
# Propiedades RDF (11 propiedades)
propiedades = {
    "riesgoAsociado": ("Inversion", "Riesgo"),
    "recomendadaPara": ("Recomendacion", "PerfilInversionista"),
    "tipoActivo": ("Inversion", "Activo"),
    "monto": ("Inversion", XSD.decimal),
    "rendimientoEsperado": ("Inversion", XSD.float),
    "edadInversionista": ("PerfilInversionista", XSD.integer),
    "nivelRiesgo": ("Riesgo", XSD.float),
    "recomienda": ("PerfilInversionista", "Recomendacion"),
    "descripcion": ("Inversion", XSD.string),
    "activoRecomendado": ("Recomendacion", "Activo"),
    "horizonte" :("Inversion",XSD.integer)
}
# Subproperty example
subproperties = {
    "activoEspecifico": "activoRelacionado",
}

for prop, (dom, ran) in propiedades.items():
    prop_uri = INV[prop]
    g.add((prop_uri, RDF.type, RDF.Property))
    g.add((prop_uri, RDFS.domain, INV[dom] if isinstance(dom, str) and ":" not in dom else FOAF[dom]))
    if isinstance(ran, URIRef):
        g.add((prop_uri, RDFS.range, ran))
    elif isinstance(ran, str) and ":" not in ran:
        g.add((prop_uri, RDFS.range, INV[ran]))
    else:
        g.add((prop_uri, RDFS.range, ran))

for child, parent in subproperties.items():
    g.add((INV[child], RDFS.subPropertyOf, INV[parent]))

# Instancias (4 por clase)
for i in range(1, 5):
    inv = INV[f"Inversion{i}"]
    g.add((inv, RDF.type, INV["InversionConservadora"]))
    g.add((inv, INV["descripcion"], Literal(f"Inversión conservadora {i}", datatype=XSD.string)))
    g.add((inv, INV["monto"], Literal(1000 * i, datatype=XSD.decimal)))
    g.add((inv, INV["rendimientoEsperado"], Literal(5 * i, datatype=XSD.float)))
    g.add((inv, INV["tipoActivo"], INV["Bonos"]))
    g.add((inv, INV["horizonte"], Literal(5 * i, datatype=XSD.integer)))

    perfil = INV[f"Inversionista{i}"]
    g.add((perfil, RDF.type, INV["PerfilInversionista"]))
    g.add((perfil, FOAF.name, Literal(f"Inversionista {i}")))
    g.add((perfil, INV["edadInversionista"], Literal(30 + i, datatype=XSD.integer)))

    riesgo = INV[f"Riesgo{i}"]
    g.add((riesgo, RDF.type, INV["Riesgo"]))
    g.add((riesgo, INV["nivelRiesgo"], Literal(3.0, datatype=XSD.float)))
    g.add((inv, INV["riesgoAsociado"], riesgo))

    recomendacion = INV[f"Recomendacion{i}"]
    g.add((recomendacion, RDF.type, INV["Recomendacion"]))
    g.add((recomendacion, INV["recomendadaPara"], perfil))
    g.add((recomendacion, INV["activoEspecifico"], INV["Bonos"]))


  * Aplicar razonamiento sobre la ontología RDFS anteriormente desarrollada usando DeductiveClosure(RDFS Semantics) de owlrl.
    * Completar de ser necesario la antología desarrollada para realizar lo siguiente:
    ** Documentar al menos dos casos de generación de nuevos hechos asociados a una jerarquía de clases (ver caso 1 de las diapositivas)

    * * Documentar la asociación de un caso de miembro de una clase desde el dominio o rango de sus propiedades (ver caso 2 de las diapositivas)
    ** Documentar un caso de nuevos hechos desde la relación
subproperty (ver caso 3 de las diapositivas)
    ** Todos los tres requerimientos anteriores, deben ser generados al usar el razonador OWL-RL usando DeductiveClosure(RDFS Semantics) sobre la ontología final de su dominio.

    * Comparar el grafo antes y después del razonamiento para evidenciar nuevas afirmaciones inferidas automáticamente.
    * Adicionalmente a lo anteriormente requerido se debe evidenciar al menos 4 nuevos hechos inferidos generados como resultado del razonamiento.

In [16]:
# Grafo original
grafo_antes = g.serialize(format="turtle").splitlines()
print(g.serialize(format="turtle"))

@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix inv: <http://example.org/inversiones#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

inv:Accion a rdfs:Class,
        rdfs:Resource ;
    rdfs:subClassOf inv:Accion,
        inv:Activo,
        inv:Inversion,
        rdfs:Resource .

inv:Activo a rdfs:Class,
        rdfs:Resource ;
    rdfs:subClassOf inv:Activo,
        inv:Inversion,
        rdfs:Resource .

inv:Bonos a inv:Activo,
        inv:Inversion,
        rdfs:Class,
        rdfs:Resource ;
    rdfs:subClassOf inv:Activo,
        inv:Bonos,
        inv:Inversion,
        rdfs:Resource .

inv:Criptoactivo a rdfs:Class,
        rdfs:Resource ;
    rdfs:subClassOf inv:Activo,
        inv:Criptoactivo,
        inv:Inversion,
        rdfs:Resource .

inv:Inversion a rdfs:Class,
        rdfs:Resource ;
    rdfs:subClassOf inv:Inversion,
        rdfs:Resource

In [17]:
# Razonamiento
DeductiveClosure(RDFS_Semantics).expand(g)

In [18]:
# Serialización después del razonamiento
grafo_despues = g.serialize(format="turtle").splitlines()
print(g.serialize(format="turtle"))




@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix inv: <http://example.org/inversiones#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

inv:Accion a rdfs:Class,
        rdfs:Resource ;
    rdfs:subClassOf inv:Accion,
        inv:Activo,
        inv:Inversion,
        rdfs:Resource .

inv:Activo a rdfs:Class,
        rdfs:Resource ;
    rdfs:subClassOf inv:Activo,
        inv:Inversion,
        rdfs:Resource .

inv:Bonos a inv:Activo,
        inv:Inversion,
        rdfs:Class,
        rdfs:Resource ;
    rdfs:subClassOf inv:Activo,
        inv:Bonos,
        inv:Inversion,
        rdfs:Resource .

inv:Criptoactivo a rdfs:Class,
        rdfs:Resource ;
    rdfs:subClassOf inv:Activo,
        inv:Criptoactivo,
        inv:Inversion,
        rdfs:Resource .

inv:Inversion a rdfs:Class,
        rdfs:Resource ;
    rdfs:subClassOf inv:Inversion,
        rdfs:Resource

Debido al tamaño del grafo no fue posible mostrarlo, por lo que vamos a mirar la diferencia entre los grafos en el formato turtle para analizar las inferencias

In [19]:
# Convertir a conjuntos de líneas para comparar
set_antes = set(grafo_antes)
set_despues = set(grafo_despues)

# Obtener nuevos triples inferidos (presentes después, no antes)
inferidos = set_despues - set_antes

# Mostrar los nuevos hechos
print("🔍 Nuevos hechos inferidos:\n")
for triple in sorted(inferidos):
    print(triple)

🔍 Nuevos hechos inferidos:

    rdfs:subPropertyOf inv:nivelRecomendacion .
foaf:name a rdf:Property,
inv:nivelRecomendacion a rdf:Property,
rdf:type a rdf:Property,
rdfs:Resource a rdfs:Resource .
rdfs:domain a rdf:Property,
rdfs:range a rdf:Property,
rdfs:subClassOf a rdf:Property,
rdfs:subPropertyOf a rdf:Property,


# Integración con modulo Experto

In [20]:
from experta import Fact, KnowledgeEngine, Rule, Field, MATCH
from rdflib import URIRef

# 1. Definimos el hecho base
class InversionFact(Fact):
    """Hecho derivado de la ontología"""
    inversion = Field(str)
    tipo = Field(str)
    rendimiento = Field(float)
    horizonte = Field(int)
    riesgo = Field(float)
    edad = Field(int)

# 2. Traductor desde RDF a hechos Experta
def traducir_inversiones_a_hechos(grafo):
    hechos = []
    for i in range(1, 5):
        inv_uri = INV[f"Inversion{i}"]
        perfil_uri = INV[f"Inversionista{i}"]
        riesgo_uri = INV[f"Riesgo{i}"]

        # Extraer hechos desde el grafo expandido
        rendimiento = _obtener_literal(grafo, inv_uri, INV["rendimientoEsperado"], float)
        horizonte = _obtener_literal(grafo, inv_uri, INV["horizonte"], int)
        edad = _obtener_literal(grafo, perfil_uri, INV["edadInversionista"], int)
        riesgo = _obtener_literal(grafo, riesgo_uri, INV["nivelRiesgo"], float)
        tipo = _inferir_tipo_inversion(grafo, inv_uri)

        # Crear hecho
        hechos.append(InversionFact(
            inversion=f"Inversion{i}",
            tipo=tipo,
            rendimiento=rendimiento,
            horizonte=horizonte,
            riesgo=riesgo,
            edad=edad
        ))
    return hechos

# Función auxiliar para extraer literales con cast seguro
def _obtener_literal(grafo, sujeto, predicado, tipo_dato):
    for _, _, o in grafo.triples((sujeto, predicado, None)):
        try:
            return tipo_dato(o)
        except:
            pass
    return None

# Inferir tipo de inversión según la clase más específica
def _inferir_tipo_inversion(grafo, uri):
    clases = {
        INV["InversionConservadora"]: "Conservadora",
        INV["InversionAgresiva"]: "Agresiva",
        INV["Activo"]: "Activo",
        INV["Bonos"]: "Bonos",
        INV["Accion"]: "Accion",
        INV["Criptoactivo"]: "Criptoactivo"
    }
    for _, _, clase in grafo.triples((uri, RDF.type, None)):
        if clase in clases:
            return clases[clase]
    return "Desconocido"